## Adversarial Extraction Arena — Training (Colab)

Same notebook for **Extractor** and **Adversary** SFT (no second notebook).

**Scripts:** `training/sft_warmup.py` · `training/sft_adversary.py` · (optional) `training/grpo_trainer.py`

**Artifacts:** `checkpoints/sft_warmup/`, `checkpoints/sft_adversary/`, `trainer_log_history.json` in each · `plots/sft_loss.png`, `plots/sft_adversary_loss.png`

### Continuing later (same Colab or new runtime)
1. Run **Clone + pull** (next cell) so you get the latest GitHub code.
2. **Install** cell if packages are missing.
3. **Corpus** cell — if `corpus.json` is missing, run the generator cell once.
4. Run **Extractor** (cell 5) if you need `checkpoints/sft_warmup`; skip if you already have it in Drive or only want the adversary.
5. Run **Adversary** (cell 6) — only needs `data/corpus.json`, not the extractor weights.
6. **Plots** cell, then **zip & download** (or Drive) → on your PC unzip into the git repo → `git push` → `huggingface-cli upload` (details in that section).

### Notes
- **GPU**: `Runtime → Change runtime type → GPU`
- **Base model:** `unsloth/Qwen2.5-1.5B-Instruct`
- Colab **deletes `/content`** when the runtime ends — always zip/Drive **before** disconnect.


In [ ]:
# Clone once, then `git pull` every time you continue (fresh code for sft_adversary etc.)
import os
import subprocess

REPO = "openenv-adversarial-extraction-arena"
URL = "https://github.com/Hardikjha09/openenv-adversarial-extraction-arena.git"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", URL], check=True)
os.chdir(REPO)
subprocess.run(["git", "pull"], check=False)
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# Install dependencies (GPU runtime recommended)
!pip -q install -r requirements.txt

In [ ]:
# Verify corpus exists (skip generator if already present)
!python -c "import json; d=json.load(open('data/corpus.json','r',encoding='utf-8')); print('corpus_size=',len(d)); print('keys=',sorted(list(d[0].keys())))"

In [ ]:
# (Optional) Generate corpus if missing
# !PYTHONPATH=. python data/generator.py

In [ ]:
# Real training: Extractor SFT (writes checkpoints/sft_warmup + trainer_log_history.json)
!PYTHONPATH=. python training/sft_warmup.py \
  --model_name unsloth/Qwen2.5-1.5B-Instruct \
  --output_dir checkpoints/sft_warmup

In [ ]:
# Adversary SFT → checkpoints/sft_adversary/ (+ trainer_log_history.json)
# Needs data/corpus.json only (extractor checkpoint not required). Default slice [200:400] avoids
# overlapping the extractor's first-200 SFT docs; if corpus is shorter, the script falls back.
!PYTHONPATH=. python training/sft_adversary.py \
  --model_name unsloth/Qwen2.5-1.5B-Instruct \
  --output_dir checkpoints/sft_adversary \
  --start_idx 200 \
  --n_docs 200

In [ ]:
# (Optional) Real training: GRPO (writes checkpoints/grpo_extractor + trainer_log_history.json)
# !PYTHONPATH=. python training/grpo_trainer.py \
#   --model_name checkpoints/sft_warmup \
#   --output_dir checkpoints/grpo_extractor

In [ ]:
# Evidence: generate loss plots from trainer logs
!PYTHONPATH=. python plots/generate_training_plots.py

# List artifacts
!ls -la checkpoints/sft_warmup | head
!ls -la checkpoints/sft_adversary | head
!ls -la plots | head

In [ ]:
# (Optional) Quick two-model eval on holdout — needs BOTH checkpoints above. ~few min on T4 for 5 eps.
# !PYTHONPATH=. python evaluation/run_eval.py \
#   --model_path checkpoints/sft_warmup \
#   --adversary_model_path checkpoints/sft_adversary \
#   --num_episodes 5 \
#   --save_every 1

#### Save artifacts — zip & download (laptop → GitHub → HF)

1. Run the **next cell** — it zips **`checkpoints/sft_adversary/`** + **`plots/sft_adversary_loss.png`** and starts a browser **download**.
2. On your PC: move the zip from **Downloads** into your cloned repo folder (or Desktop), **Extract here** so you get:
   - `checkpoints/sft_adversary/` …
   - `plots/sft_adversary_loss.png`
3. **GitHub:** this repo **ignores** `checkpoints/` and `*.safetensors`. To still push the adapter:
   ```bash
   git add -f checkpoints/sft_adversary/
   git add plots/sft_adversary_loss.png
   git commit -m "Add adversary Colab checkpoint and loss plot"
   git push origin main
   ```
   If any file is **>100MB**, prefer **Hugging Face only** for weights (step below), not GitHub.
4. **Hugging Face:** from repo root, after `huggingface-cli login`:
   ```bash
   huggingface-cli upload HardikJha/adversary-aea checkpoints/sft_adversary . --repo-type model
   huggingface-cli upload HardikJha/adversary-aea plots/sft_adversary_loss.png plots/sft_adversary_loss.png --repo-type model
   ```
   Or use the **model page → Files → Add file**. Paste **`artifacts/hf_adversary_model_README.md`** into the model README on the site.

#### Optional — Google Drive

If you prefer Drive instead of zip, use the **Drive** cell after the zip cell (skip zip if you only use Drive).

In [ ]:
# Zip adversary checkpoint + plot → browser download (then unzip into your local git repo)
import shutil
from pathlib import Path
from datetime import datetime

from google.colab import files

REPO = Path("/content/openenv-adversarial-extraction-arena")
if not REPO.is_dir():
    raise FileNotFoundError("Run the clone cell first.")

stamp = datetime.now().strftime("%Y%m%d_%H%M")
bundle = f"aea_adversary_colab_{stamp}"
staging = Path("/content") / bundle
staging.mkdir(parents=True, exist_ok=True)


def copy_into_staging(rel: str) -> None:
    src = REPO / rel
    if not src.exists():
        print("SKIP (missing):", rel)
        return
    dst = staging / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    print("OK:", rel)


copy_into_staging("checkpoints/sft_adversary")
copy_into_staging("plots/sft_adversary_loss.png")
# copy_into_staging("checkpoints/sft_warmup")  # uncomment if you trained extractor on this runtime

zip_path = shutil.make_archive(str(Path("/content") / bundle), "zip", "/content", bundle)
print("Download:", zip_path)
files.download(zip_path)

In [ ]:
# Optional — Copy artifacts to Google Drive (skip if you used zip download above)
from pathlib import Path
import shutil
from datetime import datetime

REPO = Path("/content/openenv-adversarial-extraction-arena")
if not REPO.is_dir():
    raise FileNotFoundError(f"Expected repo at {REPO} — run clone cell first.")

from google.colab import drive
drive.mount("/content/drive")

stamp = datetime.now().strftime("%Y%m%d_%H%M")
dest_root = Path("/content/drive/MyDrive/aea_colab_backups") / stamp
dest_root.mkdir(parents=True, exist_ok=True)


def copy_if_exists(src_rel, dest_subpath=None):
    src = REPO / src_rel
    if not src.exists():
        print("SKIP (missing):", src_rel)
        return
    dst = dest_root / (dest_subpath or src_rel)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    print("OK:", src_rel, "->", dst)


copy_if_exists("checkpoints/sft_adversary")
copy_if_exists("plots/sft_adversary_loss.png", Path("plots") / "sft_adversary_loss.png")
# copy_if_exists("checkpoints/sft_warmup")  # uncomment if extractor was trained here

print("\nBacked up under:", dest_root)

#### Optional — Upload from Colab (skip if you use laptop `huggingface-cli upload` above)

1. **Secrets** → **`HF_TOKEN`**.  
2. Run the next cell.  
3. Model README: paste **`artifacts/hf_adversary_model_README.md`** on the Hub.

In [ ]:
# Step 2 — Push checkpoints/sft_adversary + loss plot to Hugging Face
!pip -q install -U "huggingface_hub>=0.20.0"

import os
from pathlib import Path

REPO = Path("/content/openenv-adversarial-extraction-arena")
os.chdir(REPO)

from huggingface_hub import HfApi, login, create_repo

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("Add HF_TOKEN in Colab Secrets (or export HF_TOKEN)")

login(token=token)
api = HfApi(token=token)
REPO_ID = "HardikJha/adversary-aea"

create_repo(REPO_ID, repo_type="model", exist_ok=True)

api.upload_folder(
    folder_path=str(REPO / "checkpoints/sft_adversary"),
    repo_id=REPO_ID,
    repo_type="model",
)

plot_path = REPO / "plots/sft_adversary_loss.png"
if plot_path.is_file():
    api.upload_file(
        path_or_fileobj=str(plot_path),
        path_in_repo="plots/sft_adversary_loss.png",
        repo_id=REPO_ID,
        repo_type="model",
    )
    print("Uploaded plots/sft_adversary_loss.png")
else:
    print("SKIP plot (run generate_training_plots.py first):", plot_path)

print("Done:", "https://huggingface.co/" + REPO_ID)